# 01 — Documents and rules

Two jobs: convert the downloaded HTML into markdown, and write out my ~15 rules with their
exact wording.

**Mostly reading, not coding.** That's normal. This is the most valuable day in the project —
everything downstream inherits the errors I make here.

Download by hand first. See `data/SOURCES.md`.

In [3]:
import os, sys, json, re
from pathlib import Path

# works locally and in Colab
for candidate in (Path.cwd(), Path.cwd().parent, Path("/content/pa-appeal")):
    if (candidate / "data").exists():
        os.chdir(candidate)
        break
ROOT = Path.cwd()
print("working from:", ROOT)


working from: /content/pa-appeal


## HTML -> markdown

In [16]:
import re
from pathlib import Path

from bs4 import BeautifulSoup
from markdownify import markdownify


RAW_DIR = Path("data/raw")
POLICY_DIR = Path("data/policies")
POLICY_DIR.mkdir(parents=True, exist_ok=True)

html_files = sorted(RAW_DIR.glob("*.html"))

print("HTML files found:", len(html_files))


for source in html_files:
    html = source.read_text(
        encoding="utf-8",
        errors="replace",
    )

    soup = BeautifulSoup(html, "html.parser")

    # Remove code, styling, icons, and page navigation.
    for tag in soup.find_all([
        "script",
        "style",
        "noscript",
        "svg",
        "nav",
        "header",
        "footer",
    ]):
        tag.decompose()

    main = soup.find("main") or soup.body or soup

    markdown = markdownify(
        str(main),
        heading_style="ATX",
    )

    # Find the CMS document title.
    title_match = re.search(
        r"(?m)^#(?!#)\s+\S.*$",
        markdown,
    )

    if not title_match:
        raise ValueError(
            f"Could not find a document title in {source.name}"
        )

    # Remove everything before the title.
    markdown = markdown[title_match.start():]

    # Find the beginning of the real document.
    section_match = re.search(
        r"(?m)^##\s+(?:"
        r"Contractor Information|"
        r"LCD Information|"
        r"Article Information|"
        r"NCD Information|"
        r"Tracking Information"
        r")\s*$",
        markdown,
    )

    if not section_match:
        # Fallback to the first level-two heading.
        section_match = re.search(
            r"(?m)^##(?!#)\s+\S.*$",
            markdown,
        )

    if section_match:
        title = markdown.splitlines()[0].strip()

        markdown = (
            title
            + "\n\n"
            + markdown[section_match.start():]
        )

    # Remove anything beginning with the CMS Page Help section.
    help_match = re.search(
        r"(?mi)^##\s+Page Help\b",
        markdown,
    )

    if help_match:
        markdown = markdown[:help_match.start()]

    # Remove dialogs and licensed boilerplate appended after the document.
    end_match = re.search(
        r"(?mi)^("
        r"Read the \[(?:LCD|Article) Disclaimer\]"
        r"|CPT Copyright Statement"
        r"|Email this document to yourself or someone else"
        r"|## License For Use Of"
        r")",
        markdown,
    )

    if end_match:
        markdown = markdown[:end_match.start()]

    # Remove remaining Expand/Collapse controls.
    markdown = re.sub(
        r"(?m)^\[(?:Expand All|Collapse All)\]"
        r"\([^\n]+\)\s*\|?\s*$",
        "",
        markdown,
    )

    # Remove tracking parameters from links.
    markdown = markdown.replace(
        "&utm_source=chatgpt.com",
        "",
    )

    markdown = markdown.replace(
        "?utm_source=chatgpt.com",
        "",
    )

    # Collapse excessive blank lines.
    markdown = re.sub(
        r"\n{3,}",
        "\n\n",
        markdown,
    ).strip()

    output = POLICY_DIR / f"{source.stem}.md"

    output.write_text(
        markdown,
        encoding="utf-8",
    )

    print(
        f"{source.name:<22} -> "
        f"{output.name:<22} "
        f"{len(markdown):>8,} characters"
    )

HTML files found: 13
A52467.html            -> A52467.md                33,665 characters
A55426.html            -> A55426.md                83,221 characters
L33370.html            -> L33370.md               123,423 characters
L33718.html            -> L33718.md                42,465 characters
L33788.html            -> L33788.md                24,624 characters
L33789.html            -> L33789.md                52,494 characters
L33794.html            -> L33794.md                64,534 characters
L33797.html            -> L33797.md                47,782 characters
L33820.html            -> L33820.md                23,388 characters
L33822.html            -> L33822.md                44,383 characters
L33831.html            -> L33831.md                52,191 characters
NCD240.4.1.html        -> NCD240.4.1.md             6,522 characters
NCD240.4.html          -> NCD240.4.md              13,546 characters


In [17]:
from pathlib import Path


EXPECTED = {
    "A52467.md": "A52467",
    "A55426.md": "A55426",
    "L33370.md": "L33370",
    "L33718.md": "L33718",
    "L33788.md": "L33788",
    "L33789.md": "L33789",
    "L33794.md": "L33794",
    "L33797.md": "L33797",
    "L33820.md": "L33820",
    "L33822.md": "L33822",
    "L33831.md": "L33831",
    "NCD240.4.md": "240.4",
    "NCD240.4.1.md": "240.4.1",
}

FORBIDDEN = [
    ".svg-inline--fa",
    "<script",
    "License For Use Of",
    "CPT Copyright Statement",
    "Email this document to yourself",
    "Your Name:",
    "I Do Not Accept",
]

policies = sorted(
    Path("data/policies").glob("*.md")
)

problems = []

print("Policy count:", len(policies))

for path in policies:
    text = path.read_text(encoding="utf-8")
    expected_id = EXPECTED.get(path.name)

    checks = {
        "known_file": expected_id is not None,
        "has_title": text.startswith("# "),
        "has_id": (
            expected_id in text
            if expected_id
            else False
        ),
        "enough_text": len(text) > 1_000,
        "clean": not any(
            item.lower() in text.lower()
            for item in FORBIDDEN
        ),
    }

    failed = [
        name
        for name, passed in checks.items()
        if not passed
    ]

    if failed:
        problems.append(
            (path.name, failed)
        )

    last_line = next(
        (
            line
            for line in reversed(text.splitlines())
            if line.strip()
        ),
        "EMPTY",
    )

    print(
        f"{path.name:<18}",
        f"{len(text):>8,} chars",
        f"failed={failed or 'none'}",
        f"last={last_line[:60]}",
    )

assert len(policies) == 13, (
    f"Expected 13 files, found {len(policies)}"
)

assert not problems, problems

print("All 13 documents passed the stronger validation.")

Policy count: 13
A52467.md            33,665 chars failed=none last=N/A
A55426.md            83,221 chars failed=none last=N/A
L33370.md           123,423 chars failed=none last=N/A
L33718.md            42,465 chars failed=none last=N/A
L33788.md            24,624 chars failed=none last=N/A
L33789.md            52,494 chars failed=none last=N/A
L33794.md            64,534 chars failed=none last=N/A
L33797.md            47,782 chars failed=none last=N/A
L33820.md            23,388 chars failed=none last=N/A
L33822.md            44,383 chars failed=none last=N/A
L33831.md            52,191 chars failed=none last=N/A
NCD240.4.1.md         6,522 chars failed=none last=| Sleep Testing for Obstructive Sleep Apnea (OSA) | 1 | 03/0
NCD240.4.md          13,546 chars failed=none last=| Continuous Positive Airway Pressure (CPAP) | 1 | 04/01/200
All 13 documents passed the stronger validation.


In [18]:
policies = {p.stem: p.read_text() for p in sorted(Path("data/policies").glob("*.md"))}
print(len(policies), "documents")   # want 13: 5 real + 8 decoys
print(sorted(policies))


13 documents
['A52467', 'A55426', 'L33370', 'L33718', 'L33788', 'L33789', 'L33794', 'L33797', 'L33820', 'L33822', 'L33831', 'NCD240.4', 'NCD240.4.1']


In [11]:
import shutil

archive = shutil.make_archive(
    base_name="/content/policies2",
    format="zip",
    root_dir="/content/pa-appeal/data",
    base_dir="policies",
)

print("Created:", archive)

Created: /content/policies2.zip


## My rules

Read L33718 with a pen. For each rule, **copy-paste** the sentence. Do not retype it.

This list does two jobs: it's how I cut the policy into chunks, and it's the answer key I check
retrieval against. Slightly-wrong text means every retrieval number is quietly wrong and I
can't tell.

Each entry also carries `phase`, `device` and `spec_fields`, so notebook 04 can ask a case only
about the criteria that apply to it, and notebook 02 knows which record fields decide each one.


In [ ]:
# criteria = [
#     {"id": "B1",
#      "summary": "AHI/RDI at least 15 with at least 30 events",
#      "text": "",                      # <- paste from L33718
#      "source": "L33718"},

#     {"id": "B2",
#      "summary": "AHI/RDI 5 to 14 with at least 10 events",
#      "text": "",
#      "source": "L33718"},

#     {"id": "B2_symptoms",
#      "summary": "one of: sleepiness, impaired cognition, mood disorder, insomnia, "
#                 "hypertension, ischemic heart disease, stroke history",
#      "text": "",
#      "source": "L33718"},

#     {"id": "apnea_def",     "summary": "airflow stops >= 10 seconds", "text": "", "source": "L33718"},
#     {"id": "hypopnea_def",  "summary": ">=10s, >=30% less airflow, >=4% O2 drop", "text": "", "source": "L33718"},
#     {"id": "reeval_window", "summary": "re-evaluation between day 31 and day 91", "text": "", "source": "L33718"},
#     {"id": "adherence",     "summary": ">=4 hrs/night on 70% of nights in a 30-day stretch", "text": "", "source": "L33718"},
#     {"id": "E0601",         "summary": "E0601 is the CPAP device", "text": "", "source": "L33718"},
#     {"id": "E0470_trial",   "summary": "E0470 for OSA only if E0601 tried and failed", "text": "", "source": "L33718"},
#     {"id": "E0471_osa",     "summary": "E0471 not covered when primary diagnosis is OSA", "text": "", "source": "L33718"},

#     # ...aim for ~15. Add whatever else L33718 actually says that my cases test.
# ]

# len(criteria)


In [1]:
import re

# Every criterion carries five things beyond the quote itself:
#
#   text         verbatim from L33718 — copy-pasted, markdown list marker and all.
#   text_core    the same sentence with the list marker and trailing connector
#                stripped. This is what model quotes get matched against.
#   phase        definition | initial | continued | general. Decides which
#                criteria a given case is even asked about.
#   device       HCPCS codes this criterion applies to (None = all of them).
#   spec_fields  the patient-record fields that decide it — the bridge to
#                oracle() in notebook 02.
#
# Convention: "met" always means "this supports coverage". L33718 states the
# E0471 rule as a prohibition, so its summary is restated as a requirement —
# otherwise the model reads "met" as "the rule was obeyed" and the answer key
# and the prediction mean opposite things.


def core(text):
    """Strip the list marker and trailing connector from a verbatim quote.

    The leading "1. " prefixes are markdownify artifacts, and the source
    numbering is misleading anyway (device_instruction is criterion C,
    E0470_trial is D). A model quoting the sentence drops them, so matching on
    the raw text scores real quotes as invented. Stripping from both ends keeps
    text_core a contiguous substring of the policy, so it still verifies.
    """
    text = re.sub(r"^\d+\.\s+", "", text)
    return re.sub(r"\s*;\s*(and|or),?$", "", text).strip()


criteria = [
    {
        "id": "apnea_def",
        "summary": "Apnea means airflow stops for at least 10 seconds",
        "text": (
            "Apnea is defined as the cessation of airflow for at least 10 seconds."
        ),
        "source": "L33718",
        "phase": "definition",
        "device": None,
        "spec_fields": [],
    },
    {
        "id": "hypopnea_def",
        "summary": (
            "Hypopnea lasts at least 10 seconds, reduces airflow by at least 30%, and "
            "lowers oxygen saturation by at least 4%"
        ),
        "text": (
            "Hypopnea is defined as an abnormal respiratory event lasting at least 10 "
            "seconds associated with at least a 30% reduction in thoracoabdominal "
            "movement or airflow as compared to baseline, and with at least a 4% "
            "decrease in oxygen saturation."
        ),
        "source": "L33718",
        "phase": "definition",
        "device": None,
        "spec_fields": [],
    },
    {
        "id": "ahi_def",
        "summary": (
            "AHI is apneas plus hypopneas per hour of sleep, RERAs excluded, measurable "
            "only in a Type I or Type II study"
        ),
        "text": (
            "The apnea-hypopnea index (AHI) is defined as the average number of "
            "episodes of apnea and hypopnea per hour of sleep without the use of a "
            "positive airway pressure device. For purposes of this policy, respiratory "
            "effort related arousals (RERAs) are not included in the calculation of the "
            "AHI. Sleep time can only be measured in a Type I (facility based "
            "polysomnogram) or Type II sleep study (see descriptions below)."
        ),
        "source": "L33718",
        "phase": "definition",
        "device": None,
        "spec_fields": [],
    },
    {
        "id": "rdi_def",
        "summary": (
            "RDI is apneas plus hypopneas per hour of recording, RERAs excluded, "
            "reported by home sleep studies"
        ),
        "text": (
            "The respiratory disturbance index (RDI) is defined as the average number "
            "of apneas plus hypopneas per hour of recording without the use of a "
            "positive airway pressure device. For purposes of this policy, respiratory "
            "effort related arousals (RERAs) are not included in the calculation of the "
            "RDI. The RDI is reported in Type III, Type IV, and other home sleep "
            "studies."
        ),
        "source": "L33718",
        "phase": "definition",
        "device": None,
        "spec_fields": [],
    },
    {
        "id": "pap_device_codes",
        "summary": (
            "PAP includes E0601 single-level CPAP and E0470 bi-level without backup "
            "rate"
        ),
        "text": (
            "In this policy, the term PAP (positive airway pressure) device will refer "
            "to both a single-level continuous positive airway pressure device (E0601) "
            "and a bi-level respiratory assist device without back-up rate (E0470) when "
            "it is used in the treatment of obstructive sleep apnea."
        ),
        "source": "L33718",
        "phase": "definition",
        "device": None,
        "spec_fields": [],
    },
    {
        "id": "initial_evaluation",
        "summary": (
            "An in-person clinical evaluation must occur before the sleep test"
        ),
        "text": (
            "1. The beneficiary has an in-person clinical evaluation by the treating "
            "practitioner prior to the sleep test to assess the beneficiary for "
            "obstructive sleep apnea."
        ),
        "source": "L33718",
        "phase": "initial",
        "device": ["E0601", "E0470"],
        "spec_fields": ["initial_eval_before_test"],
    },
    {
        "id": "B1",
        "summary": "AHI or RDI at least 15 per hour with at least 30 events",
        "text": (
            "1. The apnea-hypopnea index (AHI) or Respiratory Disturbance Index (RDI) "
            "is greater than or equal to 15 events per hour with a minimum of 30 "
            "events; or,"
        ),
        "source": "L33718",
        "phase": "initial",
        "device": ["E0601", "E0470"],
        "spec_fields": ["ahi", "events"],
    },
    {
        "id": "B2",
        "summary": (
            "AHI or RDI from 5 through 14 with at least 10 events and a qualifying "
            "symptom or condition"
        ),
        "text": (
            "2. The AHI or RDI is greater than or equal to 5 and less than or equal to "
            "14 events per hour with a minimum of 10 events and documentation of:\n"
            "\n"
            "1. Excessive daytime sleepiness, impaired cognition, mood disorders, or "
            "insomnia; or,\n"
            "2. Hypertension, ischemic heart disease, or history of stroke."
        ),
        "source": "L33718",
        "phase": "initial",
        "device": ["E0601", "E0470"],
        "spec_fields": ["ahi", "events", "symptom"],
    },
    {
        "id": "short_study_events",
        "summary": (
            "Studies under two hours still require 30 events without symptoms or 10 "
            "events with symptoms"
        ),
        "text": (
            "If the AHI or RDI is calculated based on less than 2 hours of sleep or "
            "recording time, the total number of recorded events used to calculate the "
            "AHI or RDI (respectively) must be at least the number of events that would "
            "have been required in a 2 hour period (i.e., must reach ≥30 events without "
            "symptoms or ≥10 events with symptoms)."
        ),
        "source": "L33718",
        "phase": "initial",
        "device": ["E0601", "E0470"],
        "spec_fields": ["ahi", "events", "study_hours", "symptom"],
    },
    {
        "id": "sleep_test_valid",
        "summary": (
            "The sleep test is Medicare-valid, FDA-approved, ordered by the treating "
            "practitioner, and run by a qualified entity"
        ),
        "text": (
            "Coverage of a PAP device for the treatment of OSA is limited to claims "
            "where the diagnosis of OSA is based upon all of the following:\n"
            "\n"
            "1. A sleep test (Type I, II, III, IV, Other) that meets the Medicare "
            "requirements for a valid sleep test as outlined in NCD 240.4.1 and the "
            "applicable A/B MAC LCD and Billing and Coding article; and,\n"
            "2. A sleep test that is approved by the Food and Drug Administration (FDA) "
            "as a diagnostic device; and,\n"
            "3. The sleep test results meet the coverage criteria in effect for the "
            "date of service of the claim for the PAP device; and,\n"
            "4. The sleep test is ordered by the beneficiary’s treating practitioner; "
            "and,\n"
            "5. The sleep test is conducted by an entity that qualifies as a Medicare "
            "provider of sleep tests and is in compliance with all applicable state "
            "regulatory requirements."
        ),
        "source": "L33718",
        "phase": "initial",
        "device": ["E0601", "E0470"],
        # All five numbered items, as documentation of what the rule asks for.
        # make_spec() varies test_type and test_ordered_by; the rest default to
        # None, because real chart notes don't state them. meets_date_criteria
        # points back at B1/B2 -- keep it listed, but leave it out of oracle()'s
        # conjunction or the same AHI facts get counted twice in the F1.
        "spec_fields": [
            "test_type",
            "medicare_valid",
            "fda_approved",
            "meets_date_criteria",
            "test_ordered_by",
            "test_provider_qualified",
            "state_requirements_met",
        ],
    },
    {
        "id": "device_instruction",
        "summary": (
            "The beneficiary or caregiver receives instruction on device use and care"
        ),
        "text": (
            "3. The beneficiary and/or their caregiver has received instruction from "
            "the supplier of the device in the proper use and care of the equipment."
        ),
        "source": "L33718",
        "phase": "initial",
        "device": ["E0601", "E0470"],
        "spec_fields": ["instruction_given"],
    },
    {
        "id": "E0470_trial",
        "summary": "E0601 must be tried and proven ineffective before E0470",
        "text": (
            "4. An E0601 has been tried and proven ineffective based on a therapeutic "
            "trial conducted in either a facility or in a home setting."
        ),
        "source": "L33718",
        "phase": "initial",
        "device": ["E0470"],
        "spec_fields": ["e0601_tried", "e0601_ineffective"],
    },
    {
        "id": "E0470_ineffective",
        "summary": (
            "Ineffective means failure to meet therapeutic goals despite optimal E0601 "
            "therapy"
        ),
        "text": (
            "Ineffective is defined as documented failure to meet therapeutic goals "
            "using an E0601 during the titration portion of a facility-based study or "
            "during home use despite optimal therapy (i.e., proper mask selection and "
            "fitting and appropriate pressure settings)."
        ),
        "source": "L33718",
        "phase": "initial",
        "device": ["E0470"],
        "spec_fields": ["e0601_ineffective"],
    },
    {
        "id": "E0471_osa",
        "summary": (
            "The device billed is eligible for OSA coverage: "
            "E0601 or E0470, not E0471 with a backup rate"
        ),
        "text": (
            "A bi-level positive airway pressure device with back-up rate (E0471) is "
            "not reasonable and necessary if the primary diagnosis is OSA. If an E0471 "
            "is billed with a diagnosis of OSA, it will be denied as not reasonable and "
            "necessary."
        ),
        "source": "L33718",
        "phase": "initial",
        "device": ["E0471"],
        "spec_fields": ["device", "primary_dx"],
    },
    {
        "id": "reeval_window",
        "summary": (
            "Continued coverage requires reevaluation from day 31 through day 91"
        ),
        "text": (
            "Continued coverage of a PAP device (E0470 or E0601) beyond the first three "
            "months of therapy requires that, no sooner than the 31st day but no later "
            "than the 91st day after initiating therapy, the treating practitioner must "
            "conduct a clinical re-evaluation and document that the beneficiary is "
            "benefiting from PAP therapy."
        ),
        "source": "L33718",
        "phase": "continued",
        "device": ["E0601", "E0470"],
        "spec_fields": ["reeval_day"],
    },
    {
        "id": "reeval_late",
        "summary": (
            "A reevaluation after day 91 still allows coverage, beginning on the date "
            "of that reevaluation"
        ),
        "text": (
            "If the treating practitioner re-evaluation does not occur until after the "
            "91st day but the evaluation demonstrates that the beneficiary is "
            "benefiting from PAP therapy as defined in criteria 1 and 2 above, "
            "continued coverage of the PAP device will commence with the date of that "
            "re-evaluation."
        ),
        "source": "L33718",
        "phase": "continued",
        "device": ["E0601", "E0470"],
        "spec_fields": ["reeval_day",
    "symptoms_improved",
    "adherence_reviewed",
    "usage_hours",
    "usage_pct",
    "usage_window_days",],
    },
    {
        "id": "symptoms_improved",
        "summary": "Reevaluation must document improvement in OSA symptoms",
        "text": (
            "1. In-person clinical re-evaluation by the treating practitioner with "
            "documentation that symptoms of obstructive sleep apnea are improved; and,"
        ),
        "source": "L33718",
        "phase": "continued",
        "device": ["E0601", "E0470"],
        "spec_fields": ["symptoms_improved"],
    },
    {
        "id": "adherence_reviewed",
        "summary": "The practitioner must review objective adherence evidence",
        "text": (
            "2. Objective evidence of adherence to use of the PAP device, reviewed by "
            "the treating practitioner."
        ),
        "source": "L33718",
        "phase": "continued",
        "device": ["E0601", "E0470"],
        "spec_fields": ["adherence_reviewed"],
    },
    {
        "id": "adherence",
        "summary": (
            "PAP used at least four hours per night on 70% of nights during a "
            "consecutive 30-day period"
        ),
        "text": (
            "Adherence to therapy is defined as use of PAP ≥4 hours per night on 70% of "
            "nights during a consecutive thirty (30) day period anytime during the "
            "first three (3) months of initial usage."
        ),
        "source": "L33718",
        "phase": "continued",
        "device": ["E0601", "E0470"],
        "spec_fields": ["usage_hours", "usage_pct", "usage_window_days"],
    },
    {
        "id": "swo",
        "summary": (
            "A Standard Written Order reaches the supplier before the claim is "
            "submitted"
        ),
        "text": (
            "A Standard Written Order (SWO) must be communicated to the supplier before "
            "a claim is submitted. If the supplier bills for an item addressed in this "
            "policy without first receiving a completed SWO, the claim shall be denied "
            "as not reasonable and necessary."
        ),
        "source": "L33718",
        "phase": "general",
        "device": None,
        "spec_fields": ["swo_on_file"],
    },
]

for c in criteria:
    c["text_core"] = core(c["text"])

print(len(criteria), "criteria")


20 criteria


In [4]:
policy = Path("data/policies/L33718.md").read_text(encoding="utf-8")

PHASES  = {"definition", "initial", "continued", "general"}
DEVICES = {"E0601", "E0470", "E0471"}

problems = []

for c in criteria:
    checks = {
        "text not verbatim":      c["text"] in policy,
        "text_core not verbatim": c["text_core"] in policy,
        "bad phase":              c["phase"] in PHASES,
        "bad device":             c["device"] is None or set(c["device"]) <= DEVICES,
        "no spec_fields":         bool(c["spec_fields"]) or c["phase"] == "definition",
    }
    failed = [name for name, ok in checks.items() if not ok]
    print(f"{c['id']:<20} {'ok' if not failed else ', '.join(failed)}")

    if failed:
        problems.append(c["id"])

assert not problems, f"fix these before going any further: {problems}"

print(f"\nAll {len(criteria)} criteria verify against L33718.")


apnea_def            ok
hypopnea_def         ok
ahi_def              ok
rdi_def              ok
pap_device_codes     ok
initial_evaluation   ok
B1                   ok
B2                   ok
short_study_events   ok
sleep_test_valid     ok
device_instruction   ok
E0470_trial          ok
E0470_ineffective    ok
E0471_osa            ok
reeval_window        ok
reeval_late          ok
symptoms_improved    ok
adherence_reviewed   ok
adherence            ok
swo                  ok

All 20 criteria verify against L33718.


In [5]:
import json

output = Path("data/criteria.json")

output.write_text(
    json.dumps(
        criteria,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

print("Saved:", output)

Saved: data/criteria.json


## Check every quote is really there\n\nAll `True` before moving on.

In [26]:
from collections import Counter

print(Counter(c["phase"] for c in criteria))
print()

# The quotes that carry a markdown list marker. Those numbers are conversion
# artifacts and the source numbering is misleading anyway (device_instruction is
# criterion C, E0470_trial is D). A model quoting the sentence drops them, so
# matching on the raw text would score real quotes as invented.
print("marker stripped for quote matching:")
for c in criteria:
    if c["text"] != c["text_core"]:
        print(f"  {c['id']:<20} {c['text'][:52]!r}")


Counter({'initial': 9, 'definition': 5, 'continued': 5, 'general': 1})

marker stripped for quote matching:
  initial_evaluation   '1. The beneficiary has an in-person clinical evaluat'
  B1                   '1. The apnea-hypopnea index (AHI) or Respiratory Dis'
  B2                   '2. The AHI or RDI is greater than or equal to 5 and '
  device_instruction   '3. The beneficiary and/or their caregiver has receiv'
  E0470_trial          '4. An E0601 has been tried and proven ineffective ba'
  symptoms_improved    '1. In-person clinical re-evaluation by the treating '
  adherence_reviewed   '2. Objective evidence of adherence to use of the PAP'


In [27]:
# What a single case actually gets asked. A case requesting an initial E0601 is
# never asked about the E0470 trial or about month-four adherence -- those rows
# would be trivially insufficient_evidence and would dilute the F1 in notebook 05.
# Definitions are indexed for retrieval but never scored.

def criteria_for(device, phase):
    return [c for c in criteria
            if c["phase"] in (phase, "general")
            and (c["device"] is None or device in c["device"])]

for device, phase in [("E0601", "initial"), ("E0470", "initial"),
                      ("E0471", "initial"), ("E0601", "continued")]:
    asked = [c["id"] for c in criteria_for(device, phase)]
    print(f"{device} {phase:<10} {len(asked):>2}  {', '.join(asked)}")


E0601 initial     7  initial_evaluation, B1, B2, short_study_events, sleep_test_valid, device_instruction, swo
E0470 initial     9  initial_evaluation, B1, B2, short_study_events, sleep_test_valid, device_instruction, E0470_trial, E0470_ineffective, swo
E0471 initial     2  E0471_osa, swo
E0601 continued   6  reeval_window, reeval_late, symptoms_improved, adherence_reviewed, adherence, swo
